In [1]:
import pandas as pd 
import numpy as np 

In [2]:
np.random.seed(42)

x1 = np.random.randint(1, 100000, 5000)
x2 = x1 * 2 + np.random.normal(0, 1000, 5000)
x3 = np.random.randint(-5000, 5000, 5000)
x4 = np.random.normal(0, 1, 5000)

noise = np.random.normal(0, 5000, 5000)

y = 3*x1 + 1.5*x2 + 0.5*x3 + 0.01*(x1**2) + noise

In [3]:
df = pd.DataFrame( np.column_stack([x1,x2,x3,x4,y]) , columns=["feature_1","feature_2","feature_3","feature4","Target"])
df.head()

,feature_1,feature_2,feature_3,feature4,Target
0,15796.0,32836.498076,-2292.0,-0.792960,2.593430e+06
1,861.0,1854.020923,981.0,-0.310116,1.117337e+04
2,76821.0,154003.442037,722.0,1.043719,1.653117e+07
3,54887.0,110112.346667,-4253.0,0.933084,-1.249295e+07
4,6266.0,10446.184141,3992.0,0.288237,4.287855e+05


In [31]:
import plotly.express as px 

X = df.drop(columns = "Target")
corelation = X.corr()
pic = px.imshow(corelation)
pic.show()

In [33]:
from statsmodels.stats.outliers_influence import variance_inflation_factor 

vif_df = pd.DataFrame()
vif_df["features"] = X.columns
vif_df["VIF"] = [  variance_inflation_factor(X.values,i)  for i in range(X.shape[1])]
vif_df

,features,VIF
0,feature_1,12894.790471
1,feature_2,12894.778703
2,feature_3,1.000059
3,feature4,1.000103


In [26]:
from sklearn.model_selection import  train_test_split

x = df.drop(columns="Target")
y = df["Target"]
x_train,x_test,y_train,y_test = train_test_split(x,y, random_state=42, test_size=0.3)

In [27]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(x_train,y_train)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [35]:
print(f"Intercept: {model.intercept_}")
print("Coefficients:")
feature_names = X.columns
for name, coef in zip(feature_names, model.coef_):
    print(f"  {name}: {coef}")

Intercept: 4267353.970022605
Coefficients:
  feature_1: 382.66433773942083
  feature_2: -213.172689043307
  feature_3: 109.83085704234237
  feature4: 436090.50389960705


In [28]:
y_pred = model.predict(x_test)

from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("R²  :", r2_score(y_test, y_pred))

RMSE: 10481705.33894667
MAE : 8444174.669435104
R²  : 0.009532758787907936


In [50]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ("polynomial", PolynomialFeatures(degree= 2,include_bias=False)),
    ("scaler" , StandardScaler()),
    ("ridge" , Ridge(alpha=1))
])

pipeline.fit(x_train,y_train)

updated_y_pred = pipeline.predict(x_test)

print("RMSE:", np.sqrt(mean_squared_error(y_test, updated_y_pred)))
print("MAE :", mean_absolute_error(y_test, updated_y_pred))
print("R²  :", r2_score(y_test, updated_y_pred))

RMSE: 10483202.522263393
MAE : 8400607.318743818
R²  : 0.009249786334630361


In [51]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

lasso_pipeline = Pipeline([
    ("polynomial" , PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler" , StandardScaler()),
    ("lasso" , Lasso(alpha=1.5))
])

lasso_pipeline.fit(x_train,y_train)

lasso_y_pred = lasso_pipeline.predict(x_test)

print("RMSE:", np.sqrt(mean_squared_error(y_test, lasso_y_pred)))
print("MAE :", mean_absolute_error(y_test, lasso_y_pred))
print("R²  :", r2_score(y_test, lasso_y_pred))

RMSE: 10481828.724641433
MAE : 8399468.526819864
R²  : 0.009509440023634297


c:\Users\RITHIK KUMAR\.conda\envs\p11venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning:

Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.004e+17, tolerance: 4.085e+13



In [69]:
updated_df = df.drop(columns = "feature_2")
from sklearn.ensemble import GradientBoostingRegressor

df_x = updated_df.drop(columns = "Target")
df_y = updated_df["Target"]
x_train,x_test,y_train,y_test = train_test_split(df_x,df_y, test_size=0.2 , random_state=42)

gd_model = GradientBoostingRegressor(
    n_estimators= 1000,
    learning_rate= 0.01,
    max_depth= 3,
    random_state= 42
)
gd_model.fit(x_train,y_train)

,loss,'squared_error'
,learning_rate,0.01
,n_estimators,1000
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [70]:
gd_pred = gd_model.predict(x_test)

print("RMSE:", np.sqrt(mean_squared_error(y_test, gd_pred)))
print("MAE:", mean_absolute_error(y_test, gd_pred))
print("R2:", r2_score(y_test, gd_pred))

RMSE: 103610.35094923257
MAE: 58129.558062360826
R2: 0.9999029076478977


In [71]:
residuals = y_test - gd_pred
import plotly.graph_objects as go 
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=gd_pred,
    y=residuals,
    mode="markers",
    name="Residuals"
))
fig.add_hline(y=0, line_dash="dash")
fig.update_layout(
    title="Residuals vs Predictions",
    xaxis_title="Predicted Values",
    yaxis_title="Residuals"
)
fig.show()

In [27]:
X = df[["feature_1","feature_3","feature4"]].values
y = df["Target"].values
X = (X - X.mean(axis=0)) / X.std(axis=0)
m,n = X.shape

w = np.zeros(n)

b = 0.0 
lr = 0.01

In [28]:
def gradient_descent(X, y, w, b, lr):
    m = len(y)

    y_hat = np.dot(X, w) + b
    error = y - y_hat

    dw = -(2/m) * np.dot(X.T, error)
    db = -(2/m) * np.sum(error)

    w = w - lr * dw
    b = b - lr * db

    loss = np.mean(error ** 2)
    return w, b, loss


In [30]:
y_mean = y.mean()
y_std = y.std()
y_scaled = (y - y_mean) / y_std

w, b = np.zeros(X.shape[1]), 0.0

for i in range(1000):
    w, b, loss = gradient_descent(X, y_scaled, w, b, lr=0.01)

    if i % 100 == 0:
        print(i, loss)


0 0.9999999999999997
100 0.9848825419485744
200 0.9846211570803985
300 0.9846166196981696
400 0.9846165406110263
500 0.9846165392267531
600 0.984616539202421
700 0.9846165392019913
800 0.984616539201984
900 0.9846165392019837


In [29]:
for i in range(100):
    w, b, loss = gradient_descent(X, y, w, b, lr=0.01)

    if i % 10 == 0:
        print(f"Epoch {i} | Loss: {loss}")
        print(f"Epoch {i} | weights: {w}")
        print(f"Epoch {i} | bias: {b}")


Epoch 0 | Loss: 119395662442905.83
Epoch 0 | weights: [-25946.34815166   2762.34849204   5477.16516508]
Epoch 0 | bias: 41991.75915611994
Epoch 10 | Loss: 117340207323644.98
Epoch 10 | weights: [-258448.7647655    27129.16545845   54467.77486795]
Epoch 10 | bias: 418382.05633399537
Epoch 20 | Loss: 115968737245271.98
Epoch 20 | weights: [-448320.62944283   46436.26518888   94337.83966759]
Epoch 20 | bias: 725920.3329343601
Epoch 30 | Loss: 115053634722811.11
Epoch 30 | weights: [-603380.16107942   61719.59932053  126784.65228897]
Epoch 30 | bias: 977201.4958215789
Epoch 40 | Loss: 114443032691235.97
Epoch 40 | weights: [-730011.45197456   73805.38030368  153189.7660152 ]
Epoch 40 | bias: 1182516.5008998057
Epoch 50 | Loss: 114035603807242.61
Epoch 50 | weights: [-833427.59113979   83352.15851986  174677.70909265]
Epoch 50 | bias: 1350273.8083952032
Epoch 60 | Loss: 113763740584558.19
Epoch 60 | weights: [-917885.50144868   90884.52195802  192163.78444603]
Epoch 60 | bias: 1487343.74250